# Global Mountain Sampling - GEE pipeline + k-means medoid reduction

Takes an elevation-class- and geography-stratified sample of **AlphaEarth**
embeddings across **all** mountain ranges in the **GMBA Inventory** on Google
Earth Engine, then condenses it to a final set of representative **medoids**.

**Stratification (sampling, sections 1-6)**
- *Geographic* - each GMBA region gets a quota proportional to a root of its
  mountain area:  `n_region prop. area ** Q_GEO`  (Q_GEO = 0.5 -> square root).
- *Elevation class* - within a region, that quota is split across Kapos classes
  proportional to a root of each class's area:  `n_class prop. class_area ** P_ELEV`.

Root weighting (exponent < 1) deliberately over-samples small regions / rare
elevation classes so the sample spans the full diversity instead of being
dominated by a few huge ranges.

**Reduction (sections 7-8)** - k-means on all 64 embedding dimensions, run
**per Kapos class**, keeping the medoid of each cluster, to yield `N_FINAL`
(=1000) representative points exported back to GEE as an asset.

**Run order:** execute top-to-bottom. Section 6 submits the exports; wait for the
Drive CSV, download it to `../data/`, then run sections 7-8.

In [ ]:
import ee, geemap
import numpy as np, pandas as pd

PROJECT = "promising-era-496715-j5"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("GEE ready")

## 1 - Configuration

Everything tunable lives here. The two root exponents control how aggressively the sample favours small regions / rare elevation bands.

In [ ]:
# Assets
GMBA_ASSET = "projects/promising-era-496715-j5/assets/GMBA_Inventory_standard_300"
ALPHA_IC   = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
ALPHA_YEAR = 2023
DEM_ASSET  = "USGS/GMTED2010"     # fully global DEM (covers high latitudes)
DEM_BAND   = "be75"

# Kapos mountain classes (elevation-only definition; mountains are already
# delimited by the GMBA polygons, so the slope/ruggedness sub-rules are dropped).
#   K1: >=4500 m   K2: 3500-4500   K3: 2500-3500
#   K4: 1500-2500  K5: 1000-1500   K6: 300-1000
#   below 300 m -> not a mountain class (masked out)
CLASS_VALUES = [1, 2, 3, 4, 5, 6]

# Sample size and stratification exponents
N_TOTAL = 50_000     # target total points (as large as practical)
Q_GEO   = 0.5        # geographic root:  n_region prop. area ** Q_GEO
P_ELEV  = 0.5        # elevation  root:  n_class  prop. class_area ** P_ELEV

# Scales
STATS_SCALE  = 10_000   # m - coarse area accounting (fast getInfo); lower for finer splits
SAMPLE_SCALE = 1_000    # m - scale of the actual stratified sampling

# Output - full sample (asset, for further use) + Drive CSV (medoid input)
OUT_PREFIX   = "global_mountain_sample"
ASSET_OUT    = f"projects/{PROJECT}/assets/{OUT_PREFIX}"
DRIVE_FOLDER = "MountAInWater"

# k-means medoid reduction (sections 7-8): final representative subset
N_FINAL      = 1000     # number of medoids in the final sample
MEDOID_ALPHA = 0.5      # per-Kapos-class quota exponent (count ** MEDOID_ALPHA)
ASSET_1000   = f"projects/{PROJECT}/assets/{OUT_PREFIX}_1000"

## 2 - Kapos elevation classes + AlphaEarth embeddings

Mountains are simply *the inside of GMBA polygons*, so the Kapos slope/ruggedness
sub-rules are not needed - we keep only the elevation breakpoints (K1 highest …
K6 lowest; below 300 m is masked out). The sampling stack is
`[kapos_class, elevation, emb_0...emb_N]`, masked to where both a Kapos class and
an AlphaEarth embedding exist (which drops ocean / sub-300 m / unmapped pixels).

In [ ]:
dem = ee.Image(DEM_ASSET).select(DEM_BAND)

# Kapos class by elevation: K1 (>=4500) highest .. K6 (300-1000); <300 m -> 0
kapos_class = (ee.Image(0)
               .where(dem.gte(300),  6)
               .where(dem.gte(1000), 5)
               .where(dem.gte(1500), 4)
               .where(dem.gte(2500), 3)
               .where(dem.gte(3500), 2)
               .where(dem.gte(4500), 1)
               .rename("kapos_class").toInt())
kapos_class = kapos_class.updateMask(kapos_class.gt(0))   # drop non-mountain (<300 m)

alpha    = (ee.ImageCollection(ALPHA_IC)
            .filterDate(f"{ALPHA_YEAR}-01-01", f"{ALPHA_YEAR}-12-31")
            .mosaic())
emb_cols = [f"emb_{i}" for i in range(len(alpha.bandNames().getInfo()))]
alpha    = alpha.rename(emb_cols)

mask  = alpha.select(0).mask().And(kapos_class.mask())   # embeddings AND a Kapos class
stack = (kapos_class
         .addBands(dem.rename("elevation"))
         .addBands(alpha)
         .updateMask(mask))

print(f"{len(emb_cols)} embedding bands - Kapos classes {CLASS_VALUES}")

## 3 - Per-region, per-class mountain area

One server-side `reduceRegions` with a grouped sum-of-pixel-area reducer gives the area (km2) of every elevation class inside every GMBA region - the only numbers we pull back from GEE.

In [ ]:
gmba = ee.FeatureCollection(GMBA_ASSET)

area_img = (ee.Image.pixelArea().divide(1e6)   # km2 per pixel
            .addBands(kapos_class)
            .updateMask(mask))

grouped = area_img.reduceRegions(
    collection = gmba,
    reducer    = ee.Reducer.sum().group(groupField=1, groupName="cls"),
    scale      = STATS_SCALE,
    tileScale  = 8,
)

feats = grouped.select(["GMBA_V2_ID", "MapName", "groups"],
                       retainGeometry=False).getInfo()["features"]

rows = []
for f in feats:
    p = f["properties"]
    for g in p.get("groups", []):
        rows.append({"region_id": p["GMBA_V2_ID"], "region": p.get("MapName"),
                     "cls": int(g["cls"]), "area_km2": g["sum"]})
stats = pd.DataFrame(rows)
stats = stats[stats["cls"].isin(CLASS_VALUES)]
print(f"{stats['region_id'].nunique()} regions with mountain pixels")
stats.head()

## 4 - Allocation via root functions

Geographic quota per region from `area ** Q_GEO`; elevation split within each region from `class_area ** P_ELEV`. Each class quota is capped at the number of pixels actually available at `SAMPLE_SCALE`.

In [ ]:
pivot = (stats.pivot_table(index=["region_id", "region"], columns="cls",
                           values="area_km2", aggfunc="sum", fill_value=0.0)
              .reindex(columns=CLASS_VALUES, fill_value=0.0))

region_area = pivot.sum(axis=1)
geo_w       = region_area ** Q_GEO
n_reg       = N_TOTAL * geo_w / geo_w.sum()

max_pix_per_km2 = 1e6 / SAMPLE_SCALE**2
quotas = {}
for (rid, name), arow in pivot.iterrows():
    nr = n_reg.loc[(rid, name)]
    w  = arow.values ** P_ELEV
    if w.sum() == 0 or nr < 1:
        continue
    alloc = nr * w / w.sum()
    cap   = np.floor(arow.values * max_pix_per_km2)        # pixels available
    q     = np.minimum(np.round(alloc), cap).astype(int)
    if q.sum() > 0:
        quotas[int(rid)] = q.tolist()

planned = sum(sum(v) for v in quotas.values())
print(f"{len(quotas)} regions - {planned:,} points planned (target {N_TOTAL:,})")
pd.Series({c: sum(v[i] for v in quotas.values())
           for i, c in enumerate(CLASS_VALUES)}, name="points").to_frame()

## 5 - Stratified sampling, region by region

`stratifiedSample` with explicit `classPoints` realises both stratifications at once. The AlphaEarth mosaic is built once and shared across all calls, so the export graph stays small.

In [ ]:
samples = []
for rid, cpts in quotas.items():
    geom = gmba.filter(ee.Filter.eq("GMBA_V2_ID", rid)).geometry()
    fc = stack.stratifiedSample(
        numPoints   = 0,
        classBand   = "kapos_class",
        classValues = CLASS_VALUES,
        classPoints = cpts,
        region      = geom,
        scale       = SAMPLE_SCALE,
        seed        = 42,
        geometries  = True,
        tileScale   = 4,
    ).map(lambda f: f.set("region_id", rid))
    samples.append(fc)

sample = ee.FeatureCollection(samples).flatten().map(lambda f: f.set({
    "longitude": f.geometry().coordinates().get(0),
    "latitude":  f.geometry().coordinates().get(1),
}))
print(f"Assembled sampling graph for {len(samples)} regions.")

## 6 - Export the full sample (GEE asset + Drive CSV)

Two batch jobs over the same stratified sample:
- **(a) GEE asset** (`ASSET_OUT`) - the full sample, reusable directly in GEE.
- **(b) Drive CSV** - the input for the k-means medoid step in section 7 (a
  `FeatureCollection` of ~50 k points can't be pulled with `.getInfo()`, which
  aborts past 5 000 elements, so the medoids are computed from the CSV instead).

`Export.table.toAsset` has no column selector, so properties are trimmed with
`FeatureCollection.select` first; the point geometry is kept. If an asset already
exists the task fails - delete it with `earthengine rm <ASSET_OUT>` or change
`OUT_PREFIX`.

In [ ]:
selectors  = ["longitude", "latitude", "region_id", "kapos_class", "elevation"] + emb_cols
sample_out = sample.select(selectors)   # trim properties; point geometry is retained

# (a) full sample -> GEE asset (for further use in GEE)
task_asset = ee.batch.Export.table.toAsset(
    collection = sample_out, description = OUT_PREFIX, assetId = ASSET_OUT)
task_asset.start()

# (b) full sample -> Drive CSV (input for the medoid step in section 7)
task_csv = ee.batch.Export.table.toDrive(
    collection = sample_out, description = OUT_PREFIX + "_csv",
    folder = DRIVE_FOLDER, fileNamePrefix = OUT_PREFIX,
    fileFormat = "CSV", selectors = selectors)
task_csv.start()

print("Asset task:", task_asset.id, "->", ASSET_OUT)
print("Drive task:", task_csv.id, f"-> Drive/{DRIVE_FOLDER}/{OUT_PREFIX}.csv")
print("Monitor: https://code.earthengine.google.com/tasks")

## 7 - Reduce to 1000 medoids (k-means per Kapos class)

Condense the full sample to `N_FINAL` representative points with k-means on all
64 AlphaEarth embedding dimensions, run **per Kapos class**. Per-class quotas
follow the same root rule as the sampling (`count ** MEDOID_ALPHA`) and sum to
`N_FINAL`. Each cluster contributes its **medoid** - the real sample point
nearest to the cluster centroid in embedding space.

*Why per class, not per region:* with only 1000 points over ~291 regions you'd
get ~3 clusters/region (degenerate k-means); 6 classes keep each run large and
well-behaved, and elevation representativity is guaranteed. The geographic spread
is inherited from the already geo-stratified input sample.

**Before running:** the Drive export from section 6 must have finished; download
`{OUT_PREFIX}.csv` from the `MountAInWater` Drive folder into `../data/`.

In [ ]:
from pathlib import Path
from sklearn.cluster import KMeans

# The Drive export from section 6 must have finished; download it into ../data/.
CSV_PATH  = Path(f"../data/{OUT_PREFIX}.csv")
df        = pd.read_csv(CSV_PATH)
emb_local = [c for c in df.columns if c.startswith("emb_")]
df        = df.dropna(subset=emb_local + ["kapos_class"]).reset_index(drop=True)
print(f"{len(df):,} sample points loaded - {len(emb_local)} embedding dims")

# Per-class quota: count ** MEDOID_ALPHA, normalised to N_FINAL (largest-remainder).
counts = df["kapos_class"].value_counts().sort_index()
w      = counts ** MEDOID_ALPHA
raw    = N_FINAL * w / w.sum()
quota  = np.floor(raw).astype(int)
for c in (raw - quota).sort_values(ascending=False).index[: int(N_FINAL - quota.sum())]:
    quota[c] += 1
quota = quota.clip(upper=counts)        # never more clusters than candidates


def class_medoids(sub, k):
    """k-means on the 64 embedding dims; return the point nearest each centroid."""
    X  = sub[emb_local].to_numpy(np.float32)
    km = KMeans(n_clusters=int(k), random_state=42, n_init=10).fit(X)
    idx = [np.where(km.labels_ == c)[0][
               np.argmin(np.linalg.norm(X[km.labels_ == c] - km.cluster_centers_[c], axis=1))]
           for c in range(int(k))]
    out = sub.iloc[idx].copy()
    out["cluster_id"] = np.arange(int(k))
    return out


parts = []
for cls, k in quota.items():
    sub = df[df["kapos_class"] == cls].reset_index(drop=True)
    parts.append(class_medoids(sub, k))
medoids = pd.concat(parts, ignore_index=True)

print(f"\n{len(medoids)} medoids selected:")
print(pd.DataFrame({"candidates": counts, "medoids": quota}).to_string())

## 8 - Upload the 1000 medoids to a GEE asset

Saves a local CSV, then exports the medoids straight to a `FeatureCollection`
asset (`ASSET_1000`) with `Export.table.toAsset` from the authenticated session -
1000 features build fine client-side, so no `earthengine` CLI or GCS staging is
needed. This is the final product the app displays. If the asset already exists,
delete it first with `earthengine rm <ASSET_1000>` or change `OUT_PREFIX`.

In [ ]:
# Save a local copy …
keep       = ["longitude", "latitude", "kapos_class", "region_id",
              "elevation", "cluster_id"] + emb_local
medoid_csv = Path(f"../data/{OUT_PREFIX}_1000.csv")
medoids[keep].to_csv(medoid_csv, index=False)
print("Saved:", medoid_csv)

# … and export straight to a GEE asset from the authenticated session
# (1000 features build fine client-side; no earthengine CLI / GCS needed).
int_cols = {"kapos_class", "region_id", "cluster_id"}


def row_to_feature(r):
    props = {c: (int(r[c]) if c in int_cols else float(r[c]))
             for c in keep if c not in ("longitude", "latitude")}
    return ee.Feature(ee.Geometry.Point([float(r["longitude"]), float(r["latitude"])]), props)


fc_1000 = ee.FeatureCollection([row_to_feature(r) for _, r in medoids.iterrows()])

task = ee.batch.Export.table.toAsset(
    collection = fc_1000, description = OUT_PREFIX + "_1000", assetId = ASSET_1000)
task.start()
print("Export started:", task.id, "->", ASSET_1000)
print("Monitor: https://code.earthengine.google.com/tasks")

## 9 - (Optional) sanity-check the allocation

Quick local plots of the area accounting and planned per-class point counts - no extra GEE calls.

In [ ]:
import matplotlib.pyplot as plt

class_pts  = [sum(v[i] for v in quotas.values()) for i in range(len(CLASS_VALUES))]
class_area = pivot.sum(axis=0).values

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar([f"K{c}" for c in CLASS_VALUES], class_area, color="steelblue")
ax[0].set_title("Mountain area per Kapos class (km2)")
ax[1].bar([f"K{c}" for c in CLASS_VALUES], class_pts, color="seagreen")
ax[1].set_title(f"Planned points per class (P_ELEV={P_ELEV})")
plt.tight_layout(); plt.show()

top = (pd.Series({r: sum(q) for r, q in quotas.items()})
         .sort_values(ascending=False).head(12))
top.index = [stats.loc[stats.region_id == r, "region"].iloc[0] for r in top.index]
top.to_frame("points")

## 10 - Quick map of the 1000 medoids

Interactive map of the final medoids, coloured by Kapos class. Uses the
in-memory `medoids` (section 7) or falls back to `../data/{OUT_PREFIX}_1000.csv`.

In [ ]:
import geemap

# Use the in-memory medoids if section 7 was run, else load the saved CSV.
try:
    med_df = medoids
except NameError:
    med_df = pd.read_csv(f"../data/{OUT_PREFIX}_1000.csv")

KAPOS_HEX = ["#54278f", "#2171b5", "#08519c", "#1a9641", "#fee08b", "#f46d43"]

# Build an ee.FeatureCollection (hex colours work via addLayer, unlike marker icons).
fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([float(r.longitude), float(r.latitude)]),
               {"kapos_class": int(r.kapos_class)})
    for r in med_df.itertuples()
])

m = geemap.Map(center=[20, 0], zoom=2)
m.add_basemap("CartoDB.DarkMatter")
for i, c in enumerate(CLASS_VALUES):
    m.addLayer(fc.filter(ee.Filter.eq("kapos_class", c)),
               {"color": KAPOS_HEX[i]}, f"K{c}")
m.add_legend(title="Kapos class",
             keys=[f"K{c}" for c in CLASS_VALUES], colors=KAPOS_HEX)
m